# Original vs Generated summary plots 



In [ ]:
# =========================
# Global parameters
# =========================
from pathlib import Path

model_tag = "flowmatching_euler"
sampled_cells_npy_dir = Path(f"../analysis_demo/{model_tag}/sampled_cells_npy")
label_file = Path("your_label_xlsx_datapath")               # diagnosis / other labels
aab_summary_file = Path("your_aab_summary_xlsx_datapath")  # AAb labels only
original_train_h5ad = Path("your_original_train_h5ad_datapath")
original_test_h5ad = Path("your_original_test_h5ad_datapath")
output_root = Path("./generated_summary_plots_selected_bundle") / model_tag
output_root.mkdir(parents=True, exist_ok=True)
selected_transfer_bundle_dir = Path("./shared_selected_marker_transfer_bundle")
selected_transfer_bundle_dir.mkdir(parents=True, exist_ok=True)
rebuild_selected_transfer_bundle = False
selected_transfer_features = [
    "C-PEP",
    "PDX1",
    "Glucagon",
    "Somatostatin",
    "Pancreatic Polypeptide",
    "Ghrelin",
    "pS6",
    "pan-Cytokeratin",
    "CD44",
    "CD49F",
    "CD45",
    "EpCAM",
]

# Transfer-bundle hyperparameters
transfer_n_neighbors = 31
transfer_pca_n_components = 20
transfer_random_state = 42
transfer_umap_n_neighbors = 15
transfer_umap_min_dist = 0.3
transfer_umap_metric = "euclidean"
train_test_threshold = 146
umap_figsize = (7, 7)
point_size = 2.0
point_alpha = 0.25
dpi = 220

diagnosis_colors = {
    "T1D": "#d62728",
    "Control": "#1f77b4",
}

aab_group_colors = {
    "Control & AAb$^{-}$": "skyblue",
    "Control & AAb$^{+}$": "salmon",
    "T1D": "mediumseagreen",
}

beta_label_exact = []
beta_label_keywords = ["beta"]

hla_feature_candidates = ["HLA-ABC", "HLA_ABC", "HLAABC"]

selected_marker_panel = [
    "CD45",
    "Somatostatin",
    "EpCAM",
    "C-PEP",
    "Glucagon",
    "pan-Cytokeratin",
    "Pancreatic Polypeptide",
    "CD49F",
    "CD44",
    "PDX1",
    "pS6",
    "Ghrelin",
]

# Heatmap style
marker_heatmap_cmap = "coolwarm"
marker_heatmap_vmin = 0.0
marker_heatmap_vmax = 6.0
marker_heatmap_ticks = [0.0, 2.5, 5.0]
marker_heatmap_figsize = (8, 8)
marker_heatmap_cbar_pad = 0.14
preferred_celltype_order = [
    "PP", "acinar", "alpha", "beta", "delta", "ductal",
    "epsilon", "hybrid", "immune", "mesenchymal"
]

# Feature-distribution plots
distribution_target_features = ["C-PEP"]   # set None for all features

# Candidate columns
donor_id_candidates = ["donor_ID", "donor_id", "donor", "Donor_ID", "Donor"]
diagnosis_candidates = ["clinical_diagnosis", "diagnosis", "Clinical_diagnosis", "Diagnosis"]
samplecls_candidates = ["sampleCls", "samplecls", "SampleCls"]
celltype_candidates = ["cell_type_lvl2", "cell_type", "CellType", "celltype", "CellType_lvl2"]
aab_columns_expected = ["aab_gada", "aab_ia_2", "aab_iaa", "aab_znt8"]

print("output_root =", output_root)
print("selected_transfer_bundle_dir =", selected_transfer_bundle_dir)


## Shared helper block

Run this block first.


In [ ]:
# =========================
# Shared helpers
# =========================
import json
import re
from pathlib import Path

import anndata as ad
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
import umap


def pick_column(df: pd.DataFrame, candidates):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def normalize_diagnosis(x) -> str:
    s = str(x).strip()
    s_lower = s.lower()
    if "control" in s_lower:
        return "Control"
    if "t1d" in s_lower or "type 1" in s_lower:
        return "T1D"
    return s


def parse_hpap_numeric(donor_id: str) -> int:
    m = re.search(r"HPAP-(\d+)", str(donor_id))
    if not m:
        raise ValueError(f"Cannot parse HPAP numeric ID from donor_id={donor_id!r}")
    return int(m.group(1))


def assign_split(donor_id: str, threshold: int = 146) -> str:
    return "train" if parse_hpap_numeric(donor_id) <= threshold else "test"


def is_beta_label(label, beta_label_exact=None, beta_label_keywords=None) -> bool:
    s = str(label).strip().lower()
    beta_label_exact = beta_label_exact or []
    beta_label_keywords = beta_label_keywords or []
    if s in {str(x).strip().lower() for x in beta_label_exact}:
        return True
    for kw in beta_label_keywords:
        if str(kw).strip().lower() in s:
            return True
    return False


def read_h5ad_matrix(h5ad_path: Path):
    adata = ad.read_h5ad(h5ad_path)
    X = adata.X
    if hasattr(X, "toarray"):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float32)
    var_names = [str(x) for x in adata.var_names]
    obs = adata.obs.copy()
    return X, var_names, obs, adata


def align_to_feature_names(X: np.ndarray, in_feature_names, target_feature_names):
    in_map = {str(k): i for i, k in enumerate(in_feature_names)}
    idx = []
    missing = []
    for name in target_feature_names:
        if name not in in_map:
            missing.append(name)
        else:
            idx.append(in_map[name])
    if missing:
        raise ValueError(f"Missing features when aligning: {missing[:10]} (total {len(missing)})")
    return X[:, idx].astype(np.float32, copy=False)


def load_generated_final_npy(sampled_cells_npy_dir: Path):
    npy_files = sorted(Path(sampled_cells_npy_dir).glob("*.npy"))
    if not npy_files:
        raise FileNotFoundError(f"No .npy files found in {sampled_cells_npy_dir}")

    xs = []
    obs_rows = []
    for npy_path in npy_files:
        donor_id = npy_path.stem
        arr = np.load(npy_path)
        if arr.ndim != 2:
            raise ValueError(f"Expected 2D array in {npy_path}, got shape {arr.shape}")
        n_cells, _ = arr.shape
        xs.append(arr.astype(np.float32, copy=False))
        obs_rows.extend({"donor_id": donor_id, "cell_idx": i} for i in range(n_cells))
    X = np.concatenate(xs, axis=0)
    obs = pd.DataFrame(obs_rows)
    return X, obs


def to_bool_series(s: pd.Series) -> pd.Series:
    if s.dtype == object:
        lower = s.astype(str).str.strip().str.lower()
        return lower.isin(["1", "true", "yes", "y", "positive", "pos"])
    return s.fillna(0).astype(bool)


def build_summary_df(label_file: Path, aab_summary_file: Path, original_train_h5ad: Path = None, original_test_h5ad: Path = None):
    # Main labels
    labels = pd.read_excel(label_file).copy()
    donor_col = pick_column(labels, donor_id_candidates)
    if donor_col is None:
        raise KeyError(f"Could not find donor column in label file. Columns: {list(labels.columns)}")

    out = pd.DataFrame()
    out["donor_id"] = labels[donor_col].astype(str).str.strip()

    dx_col = pick_column(labels, diagnosis_candidates)
    out["diagnosis"] = labels[dx_col].map(normalize_diagnosis) if dx_col is not None else pd.NA

    samplecls_col = pick_column(labels, samplecls_candidates)
    out["sampleCls"] = labels[samplecls_col].astype(str).str.strip() if samplecls_col is not None else pd.NA

    # AAb labels only
    aab = pd.read_excel(aab_summary_file, sheet_name="donor").copy()
    donor_col_aab = pick_column(aab, donor_id_candidates)
    if donor_col_aab is None:
        raise KeyError(f"Could not find donor column in AAb summary file. Columns: {list(aab.columns)}")
    aab["donor_id"] = aab[donor_col_aab].astype(str).str.strip()

    missing_aab = [c for c in aab_columns_expected if c not in aab.columns]
    if len(missing_aab) == 0:
        aab_df = aab[["donor_id"] + aab_columns_expected].copy()
        for c in aab_columns_expected:
            aab_df[c] = to_bool_series(aab_df[c])
        aab_df["aab_positive"] = aab_df[aab_columns_expected].any(axis=1).astype(int)
        aab_df = aab_df[["donor_id", "aab_positive"]].drop_duplicates(subset=["donor_id"])
        out = out.merge(aab_df, on="donor_id", how="left")
    else:
        out["aab_positive"] = pd.NA

    out["split"] = out["donor_id"].map(lambda x: assign_split(x, train_test_threshold))

    # Backfill sampleCls from original h5ad if needed
    if out["sampleCls"].isna().any() and original_train_h5ad is not None and original_test_h5ad is not None:
        dfs = []
        for p in [original_train_h5ad, original_test_h5ad]:
            adata = ad.read_h5ad(p)
            obs = adata.obs.copy()
            donor_col_obs = pick_column(obs, donor_id_candidates)
            samplecls_col_obs = pick_column(obs, samplecls_candidates)
            if donor_col_obs is not None and samplecls_col_obs is not None:
                tmp = obs[[donor_col_obs, samplecls_col_obs]].drop_duplicates()
                tmp.columns = ["donor_id", "sampleCls_from_obs"]
                tmp["donor_id"] = tmp["donor_id"].astype(str).str.strip()
                tmp["sampleCls_from_obs"] = tmp["sampleCls_from_obs"].astype(str).str.strip()
                dfs.append(tmp)
        if dfs:
            fallback = pd.concat(dfs, axis=0).drop_duplicates(subset=["donor_id"])
            out = out.merge(fallback, on="donor_id", how="left")
            out["sampleCls"] = out["sampleCls"].where(out["sampleCls"].notna(), out["sampleCls_from_obs"])
            if "sampleCls_from_obs" in out.columns:
                out = out.drop(columns=["sampleCls_from_obs"])

    out["sampleCls"] = out["sampleCls"].where(out["sampleCls"].notna(), out["diagnosis"])

    cond1 = (out["sampleCls"] == "Control") & (out["aab_positive"] == 0)
    cond2 = (out["sampleCls"] == "Control") & (out["aab_positive"] == 1)
    cond3 = (out["sampleCls"] == "T1D")
    out["group"] = np.select(
        [cond1, cond2, cond3],
        ["Control & AAb$^{-}$", "Control & AAb$^{+}$", "T1D"],
        default="Other",
    )
    return out.drop_duplicates(subset=["donor_id"]).reset_index(drop=True)


def build_selected_transfer_bundle(ref_h5ad_path: Path, bundle_dir: Path, selected_features, label_col="cell_type_lvl2"):
    bundle_dir = Path(bundle_dir)
    bundle_dir.mkdir(parents=True, exist_ok=True)

    X_full, feature_names_full, obs_ref, adata_ref = read_h5ad_matrix(ref_h5ad_path)
    feature_names_full = [str(x).strip() for x in feature_names_full]

    feature_to_idx = {f: i for i, f in enumerate(feature_names_full)}
    selected_indices = []
    selected_found = []
    missing_features = []
    for feat in selected_features:
        feat = str(feat).strip()
        if feat in feature_to_idx:
            selected_indices.append(feature_to_idx[feat])
            selected_found.append(feat)
        else:
            missing_features.append(feat)

    if len(selected_indices) == 0:
        raise ValueError("No selected transfer features were found in reference h5ad var_names.")

    if label_col not in obs_ref.columns:
        raise KeyError(f"Reference h5ad is missing label column: {label_col}")

    X_model = X_full[:, selected_indices].astype(np.float32)
    y_ref = obs_ref[label_col].astype(str).to_numpy()

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_model)

    n_pca = int(min(transfer_pca_n_components, X_scaled.shape[1], max(2, X_scaled.shape[0] - 1)))
    pca = PCA(n_components=n_pca, random_state=transfer_random_state)
    Z_ref = pca.fit_transform(X_scaled)

    knn = KNeighborsClassifier(n_neighbors=transfer_n_neighbors, weights="distance")
    knn.fit(Z_ref, y_ref)

    umap_model = umap.UMAP(
        n_neighbors=transfer_umap_n_neighbors,
        min_dist=transfer_umap_min_dist,
        metric=transfer_umap_metric,
        random_state=transfer_random_state,
    )
    umap_model.fit(Z_ref)

    joblib.dump(scaler, bundle_dir / "scaler.joblib")
    joblib.dump(pca, bundle_dir / "pca.joblib")
    joblib.dump(knn, bundle_dir / "knn.joblib")
    joblib.dump(umap_model, bundle_dir / "umap.joblib")

    meta = {
        "reference_h5ad": str(ref_h5ad_path),
        "label_col": label_col,
        "full_feature_names": feature_names_full,
        "selected_transfer_features_requested": [str(x) for x in selected_features],
        "selected_transfer_features_used": selected_found,
        "selected_feature_indices": [int(x) for x in selected_indices],
        "missing_selected_features": missing_features,
        "n_neighbors": int(transfer_n_neighbors),
        "pca_n_components": int(n_pca),
        "random_state": int(transfer_random_state),
        "umap_n_neighbors": int(transfer_umap_n_neighbors),
        "umap_min_dist": float(transfer_umap_min_dist),
        "umap_metric": str(transfer_umap_metric),
        "method": "StandardScaler + PCA + weighted KNN classifier + UMAP",
    }
    with open(bundle_dir / "meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    return meta


def load_selected_transfer_bundle(bundle_dir: Path):
    bundle_dir = Path(bundle_dir)
    scaler = joblib.load(bundle_dir / "scaler.joblib")
    pca = joblib.load(bundle_dir / "pca.joblib")
    knn = joblib.load(bundle_dir / "knn.joblib")
    umap_model = joblib.load(bundle_dir / "umap.joblib")
    with open(bundle_dir / "meta.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
    return scaler, pca, knn, umap_model, meta


def ensure_transfer_bundle(bundle_dir: Path, ref_h5ad_path: Path, selected_features, rebuild=False):
    needed = [
        bundle_dir / "scaler.joblib",
        bundle_dir / "pca.joblib",
        bundle_dir / "knn.joblib",
        bundle_dir / "umap.joblib",
        bundle_dir / "meta.json",
    ]
    if rebuild or not all(p.exists() for p in needed):
        meta = build_selected_transfer_bundle(ref_h5ad_path, bundle_dir, selected_features)
    else:
        with open(bundle_dir / "meta.json", "r", encoding="utf-8") as f:
            meta = json.load(f)
    return meta


def extract_original_obs_info(obs: pd.DataFrame, summary_df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    donor_col = pick_column(obs, donor_id_candidates)
    dx_col = pick_column(obs, diagnosis_candidates)
    samplecls_col = pick_column(obs, samplecls_candidates)
    celltype_col = pick_column(obs, celltype_candidates)

    out = pd.DataFrame(index=obs.index)
    out["donor_id"] = obs[donor_col].astype(str).str.strip() if donor_col is not None else pd.NA
    out["diagnosis"] = obs[dx_col].map(normalize_diagnosis) if dx_col is not None else pd.NA
    out["sampleCls"] = obs[samplecls_col].astype(str).str.strip() if samplecls_col is not None else pd.NA
    out["actual_celltype"] = obs[celltype_col].astype(str) if celltype_col is not None else pd.NA
    out["split"] = split_name

    out = out.merge(summary_df, on="donor_id", how="left", suffixes=("", "_summary"))
    if "diagnosis_summary" in out.columns:
        out["diagnosis"] = out["diagnosis"].where(out["diagnosis"].notna(), out["diagnosis_summary"])
    if "sampleCls_summary" in out.columns:
        out["sampleCls"] = out["sampleCls"].where(out["sampleCls"].notna(), out["sampleCls_summary"])
    if "aab_positive_summary" in out.columns:
        out["aab_positive"] = out["aab_positive_summary"]
    if "group_summary" in out.columns:
        out["group"] = out["group_summary"]
    return out


def prepare_generated_df(sampled_cells_npy_dir: Path, selected_bundle_dir: Path, label_file: Path, aab_summary_file: Path, original_train_h5ad: Path, original_test_h5ad: Path):
    scaler, pca, knn, umap_model, meta = load_selected_transfer_bundle(selected_bundle_dir)
    full_feature_names = [str(x) for x in meta["full_feature_names"]]
    selected_indices = [int(x) for x in meta["selected_feature_indices"]]

    X_full, obs = load_generated_final_npy(sampled_cells_npy_dir)
    if X_full.shape[1] != len(full_feature_names):
        raise ValueError(
            f"Generated npy feature dimension {X_full.shape[1]} does not match full feature count {len(full_feature_names)} from selected bundle meta."
        )

    summary_df = build_summary_df(label_file, aab_summary_file, original_train_h5ad, original_test_h5ad)
    obs = obs.merge(summary_df, on="donor_id", how="left")
    if obs["diagnosis"].isna().any():
        missing = sorted(obs.loc[obs["diagnosis"].isna(), "donor_id"].unique().tolist())
        raise ValueError(f"Missing donor-level metadata for donors: {missing}")

    X_model = X_full[:, selected_indices].astype(np.float32)
    Z = pca.transform(scaler.transform(X_model))
    Y = umap_model.transform(Z)
    pred_label = np.asarray(knn.predict(Z), dtype=object)

    df = obs.copy()
    df["source_kind"] = "generated"
    df["pred_label"] = pred_label
    df["cell_label"] = df["pred_label"]
    df["is_beta"] = df["pred_label"].map(lambda x: is_beta_label(x, beta_label_exact, beta_label_keywords))
    df["is_hybrid"] = df["pred_label"].astype(str).str.strip().str.lower().eq("hybrid")
    df["umap1"] = Y[:, 0].astype(np.float32)
    df["umap2"] = Y[:, 1].astype(np.float32)

    for i, feat in enumerate(full_feature_names):
        df[feat] = X_full[:, i].astype(np.float32)

    return df, full_feature_names


def prepare_original_df(original_train_h5ad: Path, original_test_h5ad: Path, selected_bundle_dir: Path, label_file: Path, aab_summary_file: Path):
    scaler, pca, knn, umap_model, meta = load_selected_transfer_bundle(selected_bundle_dir)
    full_feature_names = [str(x) for x in meta["full_feature_names"]]
    selected_indices = [int(x) for x in meta["selected_feature_indices"]]
    summary_df = build_summary_df(label_file, aab_summary_file, original_train_h5ad, original_test_h5ad)

    X_train_raw, feat_train, obs_train_raw, _ = read_h5ad_matrix(original_train_h5ad)
    X_test_raw, feat_test, obs_test_raw, _ = read_h5ad_matrix(original_test_h5ad)

    X_train_full = align_to_feature_names(X_train_raw, feat_train, full_feature_names)
    X_test_full = align_to_feature_names(X_test_raw, feat_test, full_feature_names)

    X_train_model = X_train_full[:, selected_indices]
    X_test_model = X_test_full[:, selected_indices]

    Z_train = pca.transform(scaler.transform(X_train_model))
    Z_test = pca.transform(scaler.transform(X_test_model))
    Y_train = umap_model.transform(Z_train)
    Y_test = umap_model.transform(Z_test)

    pred_train = np.asarray(knn.predict(Z_train), dtype=object)
    pred_test = np.asarray(knn.predict(Z_test), dtype=object)

    obs_train = extract_original_obs_info(obs_train_raw, summary_df, split_name="train")
    obs_test = extract_original_obs_info(obs_test_raw, summary_df, split_name="test")

    df_train = obs_train.copy()
    df_train["source_kind"] = "original"
    df_train["pred_label"] = pred_train
    df_train["cell_label"] = df_train["actual_celltype"]
    df_train["is_beta"] = df_train["actual_celltype"].map(lambda x: is_beta_label(x, beta_label_exact, beta_label_keywords))
    df_train["is_hybrid"] = df_train["actual_celltype"].astype(str).str.strip().str.lower().eq("hybrid")
    df_train["umap1"] = Y_train[:, 0].astype(np.float32)
    df_train["umap2"] = Y_train[:, 1].astype(np.float32)

    df_test = obs_test.copy()
    df_test["source_kind"] = "original"
    df_test["pred_label"] = pred_test
    df_test["cell_label"] = df_test["actual_celltype"]
    df_test["is_beta"] = df_test["actual_celltype"].map(lambda x: is_beta_label(x, beta_label_exact, beta_label_keywords))
    df_test["is_hybrid"] = df_test["actual_celltype"].astype(str).str.strip().str.lower().eq("hybrid")
    df_test["umap1"] = Y_test[:, 0].astype(np.float32)
    df_test["umap2"] = Y_test[:, 1].astype(np.float32)

    for i, feat in enumerate(full_feature_names):
        df_train[feat] = X_train_full[:, i].astype(np.float32)
        df_test[feat] = X_test_full[:, i].astype(np.float32)

    df = pd.concat([df_train, df_test], axis=0, ignore_index=True)
    return df, full_feature_names


def subset_scope(df: pd.DataFrame, scope: str) -> pd.DataFrame:
    if scope == "all":
        return df.copy()
    if scope in {"train", "test"}:
        return df[df["split"] == scope].copy()
    raise ValueError(f"Unsupported scope={scope!r}")


def ensure_output_dir(root: Path, *parts):
    d = Path(root)
    for p in parts:
        d = d / p
    d.mkdir(parents=True, exist_ok=True)
    return d


def find_feature_name(feature_names, candidates):
    lower_map = {str(f).strip().lower(): f for f in feature_names}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def compute_fixed_limits(df_list, pad_ratio=0.04):
    all_df = pd.concat([d[["umap1", "umap2"]] for d in df_list if len(d) > 0], axis=0, ignore_index=True)
    x_min, x_max = float(all_df["umap1"].min()), float(all_df["umap1"].max())
    y_min, y_max = float(all_df["umap2"].min()), float(all_df["umap2"].max())
    x_pad = max((x_max - x_min) * pad_ratio, 1e-6)
    y_pad = max((y_max - y_min) * pad_ratio, 1e-6)
    return (x_min - x_pad, x_max + x_pad), (y_min - y_pad, y_max + y_pad)


def reorder_index_with_preferred(index_values, preferred_order):
    idx = [x for x in preferred_order if x in index_values]
    tail = [x for x in index_values if x not in idx]
    return idx + tail


## Build or load the selected-marker transfer bundle

In [ ]:
# Build or load the selected-marker transfer bundle
bundle_meta = ensure_transfer_bundle(
    bundle_dir=selected_transfer_bundle_dir,
    ref_h5ad_path=original_train_h5ad,
    selected_features=selected_transfer_features,
    rebuild=rebuild_selected_transfer_bundle,
)

print("Selected-marker transfer bundle ready:")
print("  dir:", selected_transfer_bundle_dir)
print("  selected_transfer_features_used:", bundle_meta.get("selected_transfer_features_used"))
print("  missing_selected_features:", bundle_meta.get("missing_selected_features"))
print("  method:", bundle_meta.get("method"))


## Block 1. Diagnosis-colored UMAP, original and generated

In [ ]:
gen_dx_df, _ = prepare_generated_df(sampled_cells_npy_dir, selected_transfer_bundle_dir, label_file, aab_summary_file, original_train_h5ad, original_test_h5ad)
orig_dx_df, _ = prepare_original_df(original_train_h5ad, original_test_h5ad, selected_transfer_bundle_dir, label_file, aab_summary_file)


In [ ]:
from matplotlib.lines import Line2D

def plot_diagnosis_umap(df, title, out_path, xlim=None, ylim=None):
    fig, ax = plt.subplots(figsize=umap_figsize, dpi=dpi)
    for label in ["Control", "T1D"]:
        sub = df[df["diagnosis"] == label]
        if len(sub) == 0:
            continue
        ax.scatter(
            sub["umap1"], sub["umap2"],
            s=point_size, alpha=point_alpha,
            c=diagnosis_colors[label],
            linewidths=0, rasterized=True
        )

    ax.set_title(title)
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")
    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)

    handles = [
        Line2D([0], [0], marker="o", linestyle="", markersize=8,
               markerfacecolor=diagnosis_colors["Control"], markeredgecolor="none", label="Control"),
        Line2D([0], [0], marker="o", linestyle="", markersize=8,
               markerfacecolor=diagnosis_colors["T1D"], markeredgecolor="none", label="T1D"),
    ]
    ax.legend(handles=handles, frameon=False, loc="best")
    ax.grid(False)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)

diag_root = ensure_output_dir(output_root, "diagnosis_umap")
for scope in ["all"]:
    d_gen = subset_scope(gen_dx_df, scope)
    d_orig = subset_scope(orig_dx_df, scope)
    xlim, ylim = compute_fixed_limits([d_gen, d_orig])
    out_gen = ensure_output_dir(diag_root, "generated", scope)
    out_orig = ensure_output_dir(diag_root, "original", scope)
    plot_diagnosis_umap(d_gen, f"Generated | diagnosis-colored UMAP ", out_gen / f"diagnosis_umap_generated_{scope}.png", xlim, ylim)
    plot_diagnosis_umap(d_orig, f"Original | diagnosis-colored UMAP ", out_orig / f"diagnosis_umap_original_{scope}.png", xlim, ylim)


## Block 2. Marker-expression heatmaps, original and generated

In [ ]:
gen_mk_df, feature_names = prepare_generated_df(sampled_cells_npy_dir, selected_transfer_bundle_dir, label_file, aab_summary_file, original_train_h5ad, original_test_h5ad)
orig_mk_df, _ = prepare_original_df(original_train_h5ad, original_test_h5ad, selected_transfer_bundle_dir, label_file, aab_summary_file)
lower_map = {str(f).strip().lower(): f for f in feature_names}
marker_features = [lower_map[m.lower()] for m in selected_marker_panel if m.lower() in lower_map]
print("Marker panel actually used:", marker_features)


In [ ]:
def plot_marker_heatmap(df, cell_label_col, features, title, out_path):
    mean_expr = df.groupby(cell_label_col)[features].mean()
    ordered_index = reorder_index_with_preferred(list(mean_expr.index), preferred_celltype_order)
    mean_expr = mean_expr.reindex(ordered_index)

    title_fs = 22
    tick_fs = 18
    cbar_label_fs = 18
    cbar_tick_fs = 16

    fig, ax = plt.subplots(figsize=(9.5, 9.0), dpi=dpi)

    im = ax.imshow(
        mean_expr.to_numpy(),
        aspect="auto",
        cmap=marker_heatmap_cmap,
        vmin=marker_heatmap_vmin,
        vmax=marker_heatmap_vmax,
    )

    ax.set_box_aspect(1)

    ax.set_xticks(np.arange(len(mean_expr.columns)))
    ax.set_xticklabels(
        mean_expr.columns,
        rotation=45,
        ha="right",
        fontsize=tick_fs
    )

    ax.set_yticks(np.arange(len(mean_expr.index)))
    ax.set_yticklabels(
        mean_expr.index,
        fontsize=tick_fs
    )

    ax.set_title(
        title,
        fontsize=title_fs,
        pad=14
    )

    ax.tick_params(axis="both", length=5, width=1.2)

    cbar = fig.colorbar(
        im,
        ax=ax,
        orientation="vertical",
        fraction=0.046,
        pad=0.045,
        shrink=0.9,
    )

    cbar.set_ticks(marker_heatmap_ticks)
    cbar.ax.tick_params(labelsize=cbar_tick_fs, length=4, width=1.2)

    cbar.ax.set_ylabel(
        "Mean expression in group",
        rotation=90,
        labelpad=16,
        fontsize=cbar_label_fs
    )

    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight", dpi=dpi)
    plt.show()
    plt.close(fig)

    return mean_expr

mk_root = ensure_output_dir(output_root, "marker_heatmap")
for scope in ["all"]:
    d_gen = subset_scope(gen_mk_df, scope)
    d_orig = subset_scope(orig_mk_df, scope)
    out_gen = ensure_output_dir(mk_root, "generated", scope)
    out_orig = ensure_output_dir(mk_root, "original", scope)

    mean_gen = plot_marker_heatmap(
        d_gen,
        "pred_label",
        marker_features,
        f"Generated | selected marker heatmap ",
        out_gen / f"marker_heatmap_generated_{scope}.png",
    )
    mean_orig = plot_marker_heatmap(
        d_orig,
        "actual_celltype",
        marker_features,
        f"Original | selected marker heatmap ",
        out_orig / f"marker_heatmap_original_{scope}.png",
    )

    mean_gen.to_csv(out_gen / f"marker_heatmap_generated_{scope}.csv")
    mean_orig.to_csv(out_orig / f"marker_heatmap_original_{scope}.csv")

## Block 3. Cell-label UMAP, original and generated

In [ ]:
gen_lu_df, _ = prepare_generated_df(sampled_cells_npy_dir, selected_transfer_bundle_dir, label_file, aab_summary_file, original_train_h5ad, original_test_h5ad)
orig_lu_df, _ = prepare_original_df(original_train_h5ad, original_test_h5ad, selected_transfer_bundle_dir, label_file, aab_summary_file)


In [ ]:
from matplotlib.lines import Line2D

CELLTYPE_COLOR_MAP = {
    "PP": "#1f77b4",          # blue
    "acinar": "#ff7f0e",      # orange
    "alpha": "#2ca02c",       # green
    "beta": "#d62728",        # red
    "delta": "#9467bd",       # purple
    "ductal": "#c49c94",      # light brown
    "epsilon": "#e377c2",     # pink
    "hybrid": "#c7c7c7",      # gray
    "immune": "#d6cf6f",      # yellow-olive
    "mesenchymal": "#9edae5", # light cyan
}

CELLTYPE_ORDER = [
    "PP",
    "acinar",
    "alpha",
    "beta",
    "delta",
    "ductal",
    "epsilon",
    "hybrid",
    "immune",
    "mesenchymal",
]

def _square_limits(xlim, ylim):
    x0, x1 = xlim
    y0, y1 = ylim
    x_mid = 0.5 * (x0 + x1)
    y_mid = 0.5 * (y0 + y1)
    span = max(x1 - x0, y1 - y0)
    half = 0.5 * span
    return (x_mid - half, x_mid + half), (y_mid - half, y_mid + half)

def plot_cell_label_umap(df, label_col, title, out_path, xlim=None, ylim=None):
    present_labels = df[label_col].dropna().astype(str).unique().tolist()
    labels = [lab for lab in CELLTYPE_ORDER if lab in present_labels]
    extra_labels = sorted([lab for lab in present_labels if lab not in CELLTYPE_ORDER])
    labels = labels + extra_labels

    fig, ax = plt.subplots(figsize=umap_figsize, dpi=dpi)
    ax.set_box_aspect(1)

    legend_handles = []
    for label in labels:
        color = CELLTYPE_COLOR_MAP.get(label, "#7f7f7f")
        sub = df[df[label_col].astype(str) == label]
        ax.scatter(
            sub["umap1"], sub["umap2"],
            s=point_size,
            alpha=point_alpha,
            c=color,
            linewidths=0,
            rasterized=True
        )
        legend_handles.append(
            Line2D(
                [0], [0],
                marker="o",
                linestyle="",
                markersize=7,
                markerfacecolor=color,
                markeredgecolor="none",
                label=label
            )
        )

    ax.set_title(title)
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")

    if xlim is not None and ylim is not None:
        xlim_sq, ylim_sq = _square_limits(xlim, ylim)
        ax.set_xlim(*xlim_sq)
        ax.set_ylim(*ylim_sq)

    ax.set_aspect("equal", adjustable="box")
    ax.legend(handles=legend_handles, frameon=False, bbox_to_anchor=(1.02, 0.5), loc="center left")
    ax.grid(False)

    fig.subplots_adjust(right=0.78)
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)

lu_root = ensure_output_dir(output_root, "label_umap")
for scope in ["all"]:
    d_gen = subset_scope(gen_lu_df, scope)
    d_orig = subset_scope(orig_lu_df, scope)
    xlim, ylim = compute_fixed_limits([d_gen, d_orig])

    out_gen = ensure_output_dir(lu_root, "generated", scope)
    out_orig = ensure_output_dir(lu_root, "original", scope)

    plot_cell_label_umap(
        d_gen,
        "pred_label",
        f"Generated | cell-label UMAP ",
        out_gen / f"label_umap_generated_{scope}.png",
        xlim,
        ylim,
    )
    plot_cell_label_umap(
        d_orig,
        "actual_celltype",
        f"Original | cell-label UMAP ",
        out_orig / f"label_umap_original_{scope}.png",
        xlim,
        ylim,
    )

## Block 4. β-cell proportion by AAb-related donor group, original and generated

In [ ]:
gen_bp_df, _ = prepare_generated_df(sampled_cells_npy_dir, selected_transfer_bundle_dir, label_file, aab_summary_file, original_train_h5ad, original_test_h5ad)
orig_bp_df, _ = prepare_original_df(original_train_h5ad, original_test_h5ad, selected_transfer_bundle_dir, label_file, aab_summary_file)
summary_debug = build_summary_df(label_file, aab_summary_file, original_train_h5ad, original_test_h5ad)
display(summary_debug[["donor_id", "sampleCls", "aab_positive", "group"]].head())

In [ ]:
def compute_beta_prop_donor_level(df):
    d = df[~df["is_hybrid"]].copy()
    donor_level = d.groupby(["donor_id", "group"]).agg(
        n_total=("cell_label", "size"),
        n_beta=("is_beta", "sum"),
    ).reset_index()
    donor_level["beta_prop"] = donor_level["n_beta"] / donor_level["n_total"]
    return donor_level

def plot_beta_prop_group_bar(group_mean, title, out_path):
    order = ["Control & AAb$^{-}$", "Control & AAb$^{+}$", "T1D"]
    group_mean = group_mean.reindex(order)
    title_fs = 26
    axis_label_fs = 22
    tick_fs = 20

    fig, ax = plt.subplots(figsize=(9, 6.5), dpi=dpi)

    colors = [aab_group_colors.get(g, "gray") for g in group_mean.index]
    x = np.arange(len(group_mean.index))

    ax.bar(
        x,
        group_mean.fillna(0).values,
        color=colors,
        width=0.8
    )

    ax.set_title(
        title,
        fontsize=title_fs,
        pad=18
    )

    ax.set_ylabel(
        "Cell Proportion",
        fontsize=axis_label_fs,
        labelpad=12
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        group_mean.index,
        rotation=45,
        ha="right",
        fontsize=tick_fs
    )

    ax.set_ylim(0, 0.35)
    ax.set_yticks(np.arange(0, 0.351, 0.05))

    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda y, _: "{:.0%}".format(y))
    )

    ax.tick_params(
        axis="y",
        labelsize=tick_fs,
        length=6,
        width=1.3
    )

    ax.tick_params(
        axis="x",
        length=6,
        width=1.3
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(1.3)
    ax.spines["bottom"].set_linewidth(1.3)

    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight", dpi=dpi)
    plt.show()
    plt.close(fig)
bp_root = ensure_output_dir(output_root, "beta_proportion_by_aab")
for scope in ["all"]:
    d_gen = subset_scope(gen_bp_df, scope)
    d_orig = subset_scope(orig_bp_df, scope)
    donor_gen = compute_beta_prop_donor_level(d_gen)
    donor_orig = compute_beta_prop_donor_level(d_orig)
    mean_gen = donor_gen.groupby("group")["beta_prop"].mean()
    mean_orig = donor_orig.groupby("group")["beta_prop"].mean()
    out_gen = ensure_output_dir(bp_root, "generated", scope)
    out_orig = ensure_output_dir(bp_root, "original", scope)
    plot_beta_prop_group_bar(mean_gen, f"Generated | Average β Cell Proportion (no hybrid) ", out_gen / f"beta_prop_generated_{scope}.png")
    plot_beta_prop_group_bar(mean_orig, f"Original | Average β Cell Proportion (no hybrid) ", out_orig / f"beta_prop_original_{scope}.png")
    donor_gen.to_csv(out_gen / f"beta_prop_generated_{scope}_donor_level.csv", index=False)
    donor_orig.to_csv(out_orig / f"beta_prop_original_{scope}_donor_level.csv", index=False)


## Block 5. HLA-ABC distribution in β-cells, original and generated

In [ ]:
gen_hla_df, feature_names = prepare_generated_df(sampled_cells_npy_dir, selected_transfer_bundle_dir, label_file, aab_summary_file, original_train_h5ad, original_test_h5ad)
orig_hla_df, _ = prepare_original_df(original_train_h5ad, original_test_h5ad, selected_transfer_bundle_dir, label_file, aab_summary_file)
hla_feature = find_feature_name(feature_names, hla_feature_candidates)
if hla_feature is None:
    raise KeyError(f"Could not find HLA-ABC feature. Tried candidates: {hla_feature_candidates}")
gen_hla_beta = gen_hla_df[gen_hla_df["is_beta"]].copy()
orig_hla_beta = orig_hla_df[orig_hla_df["is_beta"]].copy()


In [ ]:
def _get_group_arrays(df, feature_name, groups=("Control", "T1D")):
    arrays = {}
    for g in groups:
        arr = df.loc[df["diagnosis"] == g, feature_name].dropna().to_numpy()
        arrays[g] = arr
    return arrays


def _compute_group_stats(control, t1d):
    if len(control) == 0 or len(t1d) == 0:
        return None

    med_control = float(np.median(control))
    med_t1d = float(np.median(t1d))
    delta_median = med_t1d - med_control

    stats_row = {
        "group1": "Control",
        "group2": "T1D",
        "n_group1": len(control),
        "n_group2": len(t1d),
        "median_control": med_control,
        "median_t1d": med_t1d,
        "delta_median_t1d_minus_control": delta_median,
        "mean_control": float(np.mean(control)),
        "mean_t1d": float(np.mean(t1d)),
        "std_control": float(np.std(control, ddof=1)) if len(control) > 1 else np.nan,
        "std_t1d": float(np.std(t1d, ddof=1)) if len(t1d) > 1 else np.nan,
    }
    return stats_row


def plot_hla_violin(
    df,
    feature_name,
    title,
    out_path,
    stats_out_path=None,
    ylim=(0, 10),
):
    groups = ["Control", "T1D"]
    arrays_dict = _get_group_arrays(df, feature_name, groups=groups)
    available = [(g, arrays_dict[g]) for g in groups if len(arrays_dict[g]) > 0]

    if len(available) == 0:
        return None

    title_fs = 24
    axis_label_fs = 20
    tick_fs = 18
    median_text_fs = 16
    info_text_fs = 17

    fig, ax = plt.subplots(figsize=(10, 6.5), dpi=dpi)

    parts = ax.violinplot(
        [a for _, a in available],
        positions=np.arange(1, len(available) + 1),
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )

    fill_colors = {"Control": "#2ca02c", "T1D": "#ff7f0e"}
    for body, (g, _) in zip(parts["bodies"], available):
        body.set_facecolor(fill_colors.get(g, "gray"))
        body.set_edgecolor("black")
        body.set_alpha(1.0)
        body.set_linewidth(1.5)

    ax.boxplot(
        [a for _, a in available],
        positions=np.arange(1, len(available) + 1),
        widths=0.15,
        patch_artist=True,
        showfliers=False,
        boxprops=dict(facecolor="white", color="black", linewidth=1.5),
        medianprops=dict(color="black", linewidth=2.2),
        whiskerprops=dict(color="black", linewidth=1.5),
        capprops=dict(color="black", linewidth=1.5),
    )

    for i, (g, arr) in enumerate(available, start=1):
        x = np.random.normal(i, 0.04, size=len(arr))
        ax.scatter(x, arr, s=8, color="black", alpha=0.35, rasterized=True)

    stats_row = None
    if len(available) == 2 and available[0][0] == "Control" and available[1][0] == "T1D":
        control = available[0][1]
        t1d = available[1][1]

        stats_row = _compute_group_stats(control, t1d)
        med_control = stats_row["median_control"]
        med_t1d = stats_row["median_t1d"]
        delta_median = stats_row["delta_median_t1d_minus_control"]

        y_top = ylim[1]

        ax.text(
            1,
            min(med_control + 0.22, y_top - 0.5),
            f"{med_control:.2f}",
            ha="center",
            va="bottom",
            fontsize=median_text_fs,
            bbox=dict(boxstyle="round,pad=0.22", fc="white", ec="none", alpha=0.9),
        )

        ax.text(
            2,
            min(med_t1d + 0.22, y_top - 0.5),
            f"{med_t1d:.2f}",
            ha="center",
            va="bottom",
            fontsize=median_text_fs,
            bbox=dict(boxstyle="round,pad=0.22", fc="white", ec="none", alpha=0.9),
        )

        info_text = rf"$\Delta_{{median}}$(T1D-Control) = {delta_median:.2f}"
        ax.text(
            1.5,
            y_top - 0.45,
            info_text,
            ha="center",
            va="top",
            fontsize=info_text_fs,
            bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="black", alpha=0.9),
        )

        stats_row["feature"] = feature_name

    ax.set_xticks(np.arange(1, len(available) + 1))
    ax.set_xticklabels(
        [g for g, _ in available],
        fontsize=tick_fs
    )

    # ax.set_title(title, fontsize=title_fs, pad=10)
    ax.set_ylabel("Expression value", fontsize=axis_label_fs, labelpad=12)

    ax.set_ylim(*ylim)
    ax.set_yticks(np.arange(ylim[0], ylim[1] + 0.1, 2))
    ax.tick_params(axis="y", labelsize=tick_fs, length=6, width=1.2)
    ax.tick_params(axis="x", labelsize=tick_fs, length=6, width=1.2)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)

    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight", dpi=dpi)
    plt.show()
    plt.close(fig)

    if stats_out_path is not None and stats_row is not None:
        pd.DataFrame([stats_row]).to_csv(stats_out_path, index=False)

    return stats_row
hla_root = ensure_output_dir(output_root, "hla_abc_beta_distribution")

for scope in ["all"]:
    d_gen = subset_scope(gen_hla_beta, scope)
    d_orig = subset_scope(orig_hla_beta, scope)

    out_gen = ensure_output_dir(hla_root, "generated", scope)
    out_orig = ensure_output_dir(hla_root, "original", scope)

    plot_hla_violin(
        d_orig,
        hla_feature,
        f"Original | {hla_feature} in β-cells ",
        out_orig / f"hla_beta_original_{scope}.png",
        out_orig / f"hla_beta_original_{scope}_stats.csv",
        ylim=(0, 10),
    )

    plot_hla_violin(
        d_gen,
        hla_feature,
        f"Generated | {hla_feature} in β-cells ",
        out_gen / f"hla_beta_generated_{scope}.png",
        out_gen / f"hla_beta_generated_{scope}_stats.csv",
        ylim=(0, 10),
    )

## Block 6 UMAP marker Analysis

In [ ]:
from math import ceil
from mpl_toolkits.axes_grid1 import make_axes_locatable
import string

gen_marker_umap_df, feature_names = prepare_generated_df(
    sampled_cells_npy_dir,
    selected_transfer_bundle_dir,
    label_file,
    aab_summary_file,
    original_train_h5ad,
    original_test_h5ad,
)

orig_marker_umap_df, _ = prepare_original_df(
    original_train_h5ad,
    original_test_h5ad,
    selected_transfer_bundle_dir,
    label_file,
    aab_summary_file,
)

lower_map = {str(f).strip().lower(): f for f in feature_names}
marker_umap_features = [lower_map[m.lower()] for m in selected_marker_panel if m.lower() in lower_map]

print(marker_umap_features)

In [ ]:
def _square_limits(xlim, ylim):
    x0, x1 = xlim
    y0, y1 = ylim
    x_mid = 0.5 * (x0 + x1)
    y_mid = 0.5 * (y0 + y1)
    span = max(x1 - x0, y1 - y0)
    half = 0.5 * span
    return (x_mid - half, x_mid + half), (y_mid - half, y_mid + half)


def plot_marker_expression_umap_grid(
    df,
    features,
    title,
    out_path,
    xlim=None,
    ylim=None,
    ncols=3,
    point_size=1.4,
    bg_color="#d9d9d9",
    cmap="Reds",
    vmax_quantile=99.5,
):
    n_features = len(features)
    nrows = ceil(n_features / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.0 * ncols, 4.0 * nrows),
        dpi=dpi,
    )
    axes = np.array(axes).reshape(nrows, ncols)

    if xlim is not None and ylim is not None:
        xlim, ylim = _square_limits(xlim, ylim)

    letters = list(string.ascii_uppercase)

    for idx, feat in enumerate(features):
        r = idx // ncols
        c = idx % ncols
        ax = axes[r, c]

        expr = df[feat].to_numpy()
        valid = np.isfinite(expr)

        ax.scatter(
            df.loc[valid, "umap1"],
            df.loc[valid, "umap2"],
            s=point_size,
            c=bg_color,
            linewidths=0,
            rasterized=True,
            alpha=0.85,
        )

        expr_valid = expr[valid]
        vmax = np.nanpercentile(expr_valid, vmax_quantile)
        if not np.isfinite(vmax) or vmax <= 0:
            vmax = np.nanmax(expr_valid) if np.isfinite(np.nanmax(expr_valid)) else 1.0
        if vmax <= 0:
            vmax = 1.0

        sc = ax.scatter(
            df.loc[valid, "umap1"],
            df.loc[valid, "umap2"],
            s=point_size,
            c=expr_valid,
            cmap=cmap,
            vmin=0,
            vmax=vmax,
            linewidths=0,
            rasterized=True,
        )

        if xlim is not None and ylim is not None:
            ax.set_xlim(*xlim)
            ax.set_ylim(*ylim)

        ax.set_aspect("equal", adjustable="box")
        ax.set_box_aspect(1)
        ax.set_xlabel("UMAP1", fontsize=8)
        ax.set_ylabel("UMAP2", fontsize=8)
        ax.set_title(feat, fontsize=10, pad=4)
        ax.tick_params(axis="both", labelsize=7)

        ax.text(
            -0.12,
            1.05,
            letters[idx],
            transform=ax.transAxes,
            fontsize=16,
            fontweight="bold",
            va="top",
            ha="right",
        )

        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="3%", pad=0.03)
        cb = plt.colorbar(sc, cax=cax)
        cb.ax.tick_params(labelsize=7)

    for idx in range(n_features, nrows * ncols):
        r = idx // ncols
        c = idx % ncols
        fig.delaxes(axes[r, c])

    fig.suptitle(title, fontsize=16, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.985])
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)


marker_umap_root = ensure_output_dir(output_root, "marker_expression_umap")

for scope in ["all"]:
    d_gen = subset_scope(gen_marker_umap_df, scope)
    d_orig = subset_scope(orig_marker_umap_df, scope)

    xlim, ylim = compute_fixed_limits([d_gen, d_orig])

    out_gen = ensure_output_dir(marker_umap_root, "generated", scope)
    out_orig = ensure_output_dir(marker_umap_root, "original", scope)

    plot_marker_expression_umap_grid(
        d_gen,
        marker_umap_features,
        f"Generated | 12-marker expression UMAP ",
        out_gen / f"marker_expression_umap_generated_{scope}.png",
        xlim=xlim,
        ylim=ylim,
        ncols=3,
    )

    plot_marker_expression_umap_grid(
        d_orig,
        marker_umap_features,
        f"Original | 12-marker expression UMAP ",
        out_orig / f"marker_expression_umap_original_{scope}.png",
        xlim=xlim,
        ylim=ylim,
        ncols=3,
    )